In [1]:
# import libraries
import warnings
from pathlib import Path
import subprocess

import requests

import nglview as nv
from openbabel import pybel
import biotite.structure.io as strucio

In [2]:
# filter warnings
warnings.filterwarnings("ignore")
ob_log_handler = pybel.ob.OBMessageHandler()
pybel.ob.obErrorLog.SetOutputLevel(0)

In [3]:
# define paths
HERE = Path(_dh[-1])
DATA = HERE / "data"

In [4]:
# Download the protein structure in PDB format
pdb_id = "6CM4"

url = (f"https://files.rcsb.org/download/{pdb_id}.pdb")

response = requests.get(url)
response.raise_for_status()

protein_file = DATA / "drd2_6CM4.pdb"

with open(protein_file, "w") as file:
    file.write(response.text)

protein_file

PosixPath('/Users/bartrabelink/Desktop/ACMDD group 6 assignment/ACMDD_D2_Receptor/Workflow_steps/data/drd2_6CM4.pdb')

## Remove everything that is not protein

The downloaded PDB still contains the co-crystallised risperidone, waters and ions. If these are
converted along with the protein, risperidone physically blocks the binding site and any ligand
docked there clashes with it (which gives a large positive score instead of a negative one).

Only `ATOM` records are kept here. The original file is left untouched, because the docking box
further down is still derived from risperidone's coordinates.

In [5]:
# Keep only protein atoms (ATOM records); drop risperidone, waters and ions (HETATM)
protein_only_file = DATA / "drd2_6CM4_protein.pdb"

with open(protein_file) as infile, open(protein_only_file, "w") as outfile:
    kept = 0
    for line in infile:
        if line.startswith("ATOM"):
            outfile.write(line)
            kept += 1
    outfile.write("END\n")

print(f"Kept {kept} protein atom records -> {protein_only_file.name}")

Kept 3124 protein atom records -> drd2_6CM4_protein.pdb


In [6]:
def pdb_to_pdbqt(pdb_path, pdbqt_path, pH=7.4):
    """
    Convert a PDB file to a PDBQT file needed by docking programs of the AutoDock family.

    Parameters
    ----------
    pdb_path: str or pathlib.Path
        Path to input PDB file.
    pdbqt_path: str or pathlib.path
        Path to output PDBQT file.
    pH: float
        Protonation at given pH.
    """
    molecule = list(pybel.readfile("pdb", str(pdb_path)))[0]
    # add hydrogens at given pH
    molecule.OBMol.CorrectForPH(pH)
    molecule.addh()
    # add partial charges to each atom
    for atom in molecule.atoms:
        atom.OBAtom.GetPartialCharge()
    molecule.write("pdbqt", str(pdbqt_path), overwrite=True)
    return

In [7]:
# convert protein to PDBQT format
pdb_to_pdbqt(protein_only_file, DATA / "drd2_6CM4.pdbqt")

In [8]:
# define ligand SMILES for protein-ligand complex of interest
smiles = "CC1=C(C(=O)N2CCCCC2=N1)CCN3CCN(CC3)C4=C(C(=CC=C4)Cl)Cl"

In [9]:
def smiles_to_pdbqt(smiles, pdbqt_path, pH=7.4):
    """
    Convert a SMILES string to a PDBQT file needed by docking programs of the AutoDock family.

    Parameters
    ----------
    smiles: str
        SMILES string.
    pdbqt_path: str or pathlib.path
        Path to output PDBQT file.
    pH: float
        Protonation at given pH.
    """
    molecule = pybel.readstring("smi", smiles)
    # add hydrogens at given pH
    molecule.OBMol.CorrectForPH(pH)
    molecule.addh()
    # generate 3D coordinates
    molecule.make3D(forcefield="mmff94s", steps=10000)
    # add partial charges to each atom
    for atom in molecule.atoms:
        atom.OBAtom.GetPartialCharge()
    molecule.write("pdbqt", str(pdbqt_path), overwrite=True)
    return

In [10]:
# convert the ligand into PDBQT format
smiles_to_pdbqt(smiles, DATA / "top_hit_s2230_14934458.pdbqt")

In [11]:
# Load the structure and extract the co-crystallized ligand
structure = strucio.load_structure(DATA / "drd2_6CM4.pdb")
ligand_resname = "8NU"
ligand = structure[structure.res_name == ligand_resname]

# Define docking box
pocket_center = (ligand.coord.max(axis=0) + ligand.coord.min(axis=0)) / 2
pocket_size = (ligand.coord.max(axis=0) - ligand.coord.min(axis=0) + 5)

In [12]:
def run_smina(
    ligand_path, protein_path, out_path, pocket_center, pocket_size, num_poses=10, exhaustiveness=8,
    smina_exe="./smina",
):
    """
    Perform docking with Smina.

    Parameters
    ----------
    ligand_path: str or pathlib.Path
        Path to ligand PDBQT file that should be docked.
    protein_path: str or pathlib.Path
        Path to protein PDBQT file that should be docked to.
    out_path: str or pathlib.Path
        Path to which docking poses should be saved, SDF or PDB format.
    pocket_center: iterable of float or int
        Coordinates defining the center of the binding site.
    pocket_size: iterable of float or int
        Lengths of edges defining the binding site.
    num_poses: int
        Maximum number of poses to generate.
    exhaustiveness: int
        Accuracy of docking calculations.
    smina_exe: str or pathlib.Path
        Path to the Smina executable.

    Returns
    -------
    output_text: str
        The output of the Smina calculation.
    """
    output_text = subprocess.check_output(
        [
            str(smina_exe),
            "--ligand",
            str(ligand_path),
            "--receptor",
            str(protein_path),
            "--out",
            str(out_path),
            "--center_x",
            str(pocket_center[0]),
            "--center_y",
            str(pocket_center[1]),
            "--center_z",
            str(pocket_center[2]),
            "--size_x",
            str(pocket_size[0]),
            "--size_y",
            str(pocket_size[1]),
            "--size_z",
            str(pocket_size[2]),
            "--num_modes",
            str(num_poses),
            "--exhaustiveness",
            str(exhaustiveness),
        ],
        universal_newlines=True,  # needed to capture output text
    )
    return output_text

In [13]:
import os
import platform
import shutil

# On macOS, use a native Smina (conda create -n smina -c conda-forge smina),
# since the downloaded static binary only runs on Linux
smina_exe = shutil.which("smina")
if smina_exe is None and platform.system() == "Darwin":
    conda_smina = Path.home() / "miniforge3" / "envs" / "smina" / "bin" / "smina"
    if conda_smina.exists():
        smina_exe = str(conda_smina)

smina_path = HERE / "smina"
if smina_exe is None:
    if not smina_path.exists():
        download_url = "https://sourceforge.net/projects/smina/files/smina.static/download"
        response = requests.get(download_url, allow_redirects=True)
        response.raise_for_status()
        smina_path.write_bytes(response.content)
    smina_path.chmod(smina_path.stat().st_mode | 0o111)
    smina_exe = "./smina"

previous_cwd = Path.cwd()
try:
    os.chdir(smina_path.parent)
    output_text = run_smina(
        DATA / "top_hit_s2230_14934458.pdbqt",
        DATA / "drd2_6CM4.pdbqt",
        DATA / "top_hit_s2230_14934458_poses.sdf",
        pocket_center,
        pocket_size,
        smina_exe=smina_exe,
    )
finally:
    os.chdir(previous_cwd)

print(output_text)

   _______  _______ _________ _        _______ 
  (  ____ \(       )\__   __/( (    /|(  ___  )
  | (    \/| () () |   ) (   |  \  ( || (   ) |
  | (_____ | || || |   | |   |   \ | || (___) |
  (_____  )| |(_)| |   | |   | (\ \) ||  ___  |
        ) || |   | |   | |   | | \   || (   ) |
  /\____) || )   ( |___) (___| )  \  || )   ( |
  \_______)|/     \|\_______/|/    )_)|/     \|


smina is based off AutoDock Vina. Please cite appropriately.

Weights      Terms
-0.035579    gauss(o=0,_w=0.5,_c=8)
-0.005156    gauss(o=3,_w=2,_c=8)
0.840245     repulsion(o=0,_c=8)
-0.035069    hydrophobic(g=0.5,_b=1.5,_c=8)
-0.587439    non_dir_h_bond(g=-0.7,_b=0,_c=8)
1.923        num_tors_div

Using random seed: 617608738

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
1

In [14]:
(DATA / "top_hit_s2230_14934458_poses.sdf").exists()
# NBVAL_CHECK_OUTPUT

True

In [15]:
def split_sdf_file(sdf_path):
    """
    Split an SDF file into seperate files for each molecule.
    Each file is named with consecutive numbers.

    Parameters
    ----------
    sdf_path: str or pathlib.Path
        Path to SDF file that should be split.
    """
    sdf_path = Path(sdf_path)
    stem = sdf_path.stem
    parent = sdf_path.parent
    molecules = pybel.readfile("sdf", str(sdf_path))
    for i, molecule in enumerate(molecules, 1):
        molecule.write("sdf", str(parent / f"{stem}_{i}.sdf"), overwrite=True)
    return

In [16]:
split_sdf_file(DATA / "top_hit_s2230_14934458_poses.sdf")

In [17]:
docking_pose_id = 1
view = nv.show_structure_file(
    str(DATA / f"top_hit_s2230_14934458_poses_{docking_pose_id}.sdf"),
    representations=[{"params": {}, "type": "licorice"}],
)
view.add_pdbid(pdb_id)
view

NGLWidget()

In [18]:
view.render_image(trim=True, factor=2);

In [19]:
view._display_image()